# 2. Creare o recuperare un analyzer

Scegli una modalita:

- **Portale:** crea e salva un analyzer documentale in Content Understanding,
  usando il conto hotel. Copia il suo ID in `.env` e imposta `modalita = "portale"`.
- **Template:** modifica i campi qui sotto e usa `modalita = "template"`.
  Imposta in `.env` un ID nuovo per crearlo.

In [ ]:
import os
from dotenv import load_dotenv
from azure.ai.contentunderstanding import ContentUnderstandingClient
from azure.core.credentials import AzureKeyCredential

load_dotenv(".env", override=True)

import json
from pathlib import Path
from azure.ai.contentunderstanding.models import ContentAnalyzer

modalita = "template"
analyzer_id = os.environ["ANALYZER_ID"]

## Personalizza il template

Adatta le descrizioni al conto hotel. `extract` legge valori presenti nel
documento; `generate` produce la sintesi. Tieni distinti addebiti, accrediti
e saldo, senza duplicare i dati dell'intestazione ripetuta.

In [ ]:
template = {'baseAnalyzerId': 'prebuilt-document',
 'description': 'Analyzer didattico per un conto hotel con soggiorno e movimenti',
 'config': {'enableOcr': True,
            'enableLayout': True,
            'returnDetails': True,
            'estimateFieldSourceAndConfidence': True},
 'fieldSchema': {'name': 'conto_hotel',
                 'description': 'Estrarre un solo conto anche se intestazione e riferimenti si '
                                'ripetono su piu pagine',
                 'fields': {'Hotel': {'type': 'string',
                                      'method': 'extract',
                                      'description': 'Nome della struttura alberghiera che emette '
                                                     'il conto.'},
                            'Ospite': {'type': 'string',
                                       'method': 'extract',
                                       'description': "Nome e cognome dell'ospite intestatario, "
                                                      'non della struttura o del gruppo.'},
                            'NumeroConto': {'type': 'string',
                                            'method': 'extract',
                                            'description': 'Folio No., identificativo del conto. '
                                                           'Non confonderlo con Confirmation No. o '
                                                           'Booking No.'},
                            'Camera': {'type': 'string',
                                       'method': 'extract',
                                       'description': 'Numero della camera indicato come Room No.'},
                            'Arrivo': {'type': 'date',
                                       'method': 'extract',
                                       'description': 'Arrival. Interpretare le date del conto '
                                                      'statunitense come MM-DD-YY e restituire '
                                                      'YYYY-MM-DD.'},
                            'Partenza': {'type': 'date',
                                         'method': 'extract',
                                         'description': 'Departure. Interpretare le date del conto '
                                                        'statunitense come MM-DD-YY e restituire '
                                                        'YYYY-MM-DD.'},
                            'Movimenti': {'type': 'array',
                                          'description': 'Righe della tabella '
                                                         'Date/Description/Charges/Credits, '
                                                         'esclusi Total e Balance. Non duplicare '
                                                         'le righe per intestazioni ripetute.',
                                          'items': {'type': 'object',
                                                    'properties': {'Data': {'type': 'date',
                                                                            'method': 'extract',
                                                                            'description': 'Data '
                                                                                           'della '
                                                                                           'riga, '
                                                                                           'MM-DD-YY '
                                                                                           'nel '
                                                                                           'documento '
                                                                                           'e '
                                                                                           'YYYY-MM-DD '
                                                                                           'in '
                                                                                           'output.'},
                                                                   'Descrizione': {'type': 'string',
                                                                                   'method': 'extract',
                                                                                   'description': 'Descrizione '
                                                                                                  'del '
                                                                                                  'movimento '
                                                                                                  'senza '
                                                                                                  'numero '
                                                                                                  'o '
                                                                                                  'scadenza '
                                                                                                  'della '
                                                                                                  'carta.'},
                                                                   'Addebito': {'type': 'number',
                                                                                'method': 'extract',
                                                                                'description': 'Importo '
                                                                                               'nella '
                                                                                               'colonna '
                                                                                               'Charges; '
                                                                                               'assente '
                                                                                               'se '
                                                                                               'la '
                                                                                               'cella '
                                                                                               'e '
                                                                                               'vuota.'},
                                                                   'Accredito': {'type': 'number',
                                                                                 'method': 'extract',
                                                                                 'description': 'Importo '
                                                                                                'nella '
                                                                                                'colonna '
                                                                                                'Credits; '
                                                                                                'assente '
                                                                                                'se '
                                                                                                'la '
                                                                                                'cella '
                                                                                                'e '
                                                                                                'vuota.'}}}},
                            'TotaleAddebiti': {'type': 'number',
                                               'method': 'extract',
                                               'description': 'Importo della riga Total nella '
                                                              'colonna Charges. La virgola separa '
                                                              'le migliaia e il punto i decimali.'},
                            'TotaleAccrediti': {'type': 'number',
                                                'method': 'extract',
                                                'description': 'Importo della riga Total nella '
                                                               'colonna Credits. Non e il saldo da '
                                                               'pagare.'},
                            'Saldo': {'type': 'number',
                                      'method': 'extract',
                                      'description': 'Importo Balance ancora dovuto, anche quando '
                                                     'vale zero.'},
                            'Sintesi': {'type': 'string',
                                        'method': 'generate',
                                        'description': 'Riassunto in italiano di massimo tre frasi '
                                                       'sul soggiorno e sullo stato del pagamento, '
                                                       'basato solo sul documento.'}}},
 'models': {'completion': 'gpt-4.1', 'embedding': 'text-embedding-3-large'}}

## Crea o scarica l'analyzer

Il JSON viene salvato in `analyzer.json`. Se l'ID esiste gia, usa `portale`
per recuperarlo oppure scegli un nuovo ID per creare una nuova versione.

In [ ]:
with ContentUnderstandingClient(
    endpoint=os.environ["AZURE_CONTENTUNDERSTANDING_ENDPOINT"],
    credential=AzureKeyCredential(os.environ["AZURE_CONTENTUNDERSTANDING_API_KEY"]),
) as client:
    if modalita == "template":
        client.begin_create_analyzer(
            analyzer_id=analyzer_id,
            resource=ContentAnalyzer(template),
            allow_replace=False,
        ).result()
    elif modalita != "portale":
        raise ValueError("Usa template oppure portale.")
    analyzer = client.get_analyzer(analyzer_id=analyzer_id)

Path("analyzer.json").write_text(
    json.dumps(analyzer.as_dict(), ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print("Analyzer:", analyzer_id)
print("Campi:", ", ".join(analyzer.field_schema.fields))